In [1]:
print(123)

123


In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

85

In [4]:
documents = documents_llm

In [5]:
documents[10]

{'id': '20c5a1347e',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?',
 'answer': 'Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIt contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.'}

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
import json

user_prompt = json.dumps(doc)

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [13]:
result = response.output_parsed

print(result)

questions=['Is it still okay to join the course if I found it late?', 'Can I enroll after the course has already started?', 'If I join now, do I still have a chance to get a certificate?', "What do I need to do to be eligible for the certificate if I'm joining late?", "Can I still submit my project for certification if I'm coming in after the course began?"]


In [14]:
print(result.questions)

['Is it still okay to join the course if I found it late?', 'Can I enroll after the course has already started?', 'If I join now, do I still have a chance to get a certificate?', "What do I need to do to be eligible for the certificate if I'm joining late?", "Can I still submit my project for certification if I'm coming in after the course began?"]


In [15]:
from evaluation_utils import llm_structured

In [16]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I just found it now, or is it too late?', 'If I enroll late, will I still be able to get a certificate?', 'What’s the deadline for getting a certificate if I join after the course starts?', 'Do I need to submit my project before submissions close in order to receive a certificate?', 'Is it okay to start the course late, and what do I have to do to qualify for the certificate?']


In [17]:
print(result)

questions=['Can I still join the course if I just found it now, or is it too late?', 'If I enroll late, will I still be able to get a certificate?', 'What’s the deadline for getting a certificate if I join after the course starts?', 'Do I need to submit my project before submissions close in order to receive a certificate?', 'Is it okay to start the course late, and what do I have to do to qualify for the certificate?']


In [18]:
usage.input_tokens, usage.output_tokens

(207, 105)

In [19]:
from evaluation_utils import calc_price

In [20]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00047250000000000005,
 'total_cost': 0.00062775}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I just found it now, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll late, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for getting a certificate if I join after the course starts?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close in order to receive a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course late, and what do I have to do to qualify for the certificate?',
  'document': '74eb249bbf'}]

In [22]:
import pandas as pd

In [23]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I just found it...,74eb249bbf
1,"If I enroll late, will I still be able to get ...",74eb249bbf
2,What’s the deadline for getting a certificate ...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,"Is it okay to start the course late, and what ...",74eb249bbf


In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/85 [00:00<?, ?it/s]

In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

425

In [37]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.06329325000000001

In [38]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.06329325000000001

In [39]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [43]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)